In [ ]:
# ============================================================
# PHASE 2 - DATA PREPROCESSING
# ============================================================

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split


# ============================================================
# 1. PATHS
# ============================================================

# Notebook is inside open_lab/notebooks/
BASE_DIR = os.path.abspath("..")

RAW_FILE = os.path.join(
    BASE_DIR,
    "data",
    "raw",
    "worldriskindex-trend.csv"
)

PROCESSED_DIR = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

MODEL_DIR = os.path.join(
    BASE_DIR,
    "models"
)

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)


# ============================================================
# 2. LOAD DATA
# ============================================================

print("=" * 60)
print("PHASE 2 - DATA PREPROCESSING")
print("=" * 60)

print("\nLoading dataset:")
print(RAW_FILE)

df = pd.read_csv(RAW_FILE)

print("\nDataset loaded successfully.")
print("Rows   :", df.shape[0])
print("Columns:", df.shape[1])


# ============================================================
# 3. SHOW ORIGINAL COLUMNS
# ============================================================

print("\nFirst 20 columns:")

print(df.columns[:20].tolist())


# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "WRI.Country",
    "ISO3.Code",
    "Year",
    "W",
    "E",
    "V",
    "S",
    "C",
    "A"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:

    raise ValueError(
        "Missing required columns: "
        + str(missing_columns)
    )

print("\nRequired columns found.")


# ============================================================
# 5. RENAME MAIN WRI VARIABLES
# ============================================================

# Rename the short dataset names into readable names.

df = df.rename(
    columns={
        "WRI.Country": "Country",
        "ISO3.Code": "ISO3",
        "W": "WRI",
        "E": "Exposure",
        "V": "Vulnerability",
        "S": "Susceptibility",
        "C": "Coping",
        "A": "Adaptive"
    }
)

print("\nMain columns after renaming:")

print([
    "Country",
    "ISO3",
    "Year",
    "WRI",
    "Exposure",
    "Vulnerability",
    "Susceptibility",
    "Coping",
    "Adaptive"
])


# ============================================================
# 6. CONVERT MAIN VARIABLES TO NUMERIC
# ============================================================

numeric_columns = [
    "Year",
    "WRI",
    "Exposure",
    "Vulnerability",
    "Susceptibility",
    "Coping",
    "Adaptive"
]

for col in numeric_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# ============================================================
# 7. CHECK MISSING VALUES
# ============================================================

print("\nMissing values:")

print(
    df[
        numeric_columns
    ].isna().sum()
)


# ============================================================
# 8. REMOVE ROWS WITH MISSING MAIN VALUES
# ============================================================

before = len(df)

df = df.dropna(
    subset=numeric_columns
).reset_index(drop=True)

after = len(df)

print("\nRows removed:", before - after)
print("Rows remaining:", after)


# ============================================================
# 9. CREATE FIVE RISK CATEGORIES
# ============================================================

# The paper uses five risk categories:
#
# 0 = Very Low
# 1 = Low
# 2 = Medium
# 3 = High
# 4 = Very High
#
# Categories are created using quintiles.


category_names = [
    "Very Low",
    "Low",
    "Medium",
    "High",
    "Very High"
]


def create_risk_category(series):

    """
    Convert continuous risk score into
    five approximately equal-frequency categories.
    """

    return pd.qcut(
        series,
        q=5,
        labels=category_names,
        duplicates="drop"
    )


# Create categorical targets

df["WRI_Category"] = create_risk_category(
    df["WRI"]
)

df["Exposure_Category"] = create_risk_category(
    df["Exposure"]
)

df["Vulnerability_Category"] = create_risk_category(
    df["Vulnerability"]
)

df["Susceptibility_Category"] = create_risk_category(
    df["Susceptibility"]
)


# ============================================================
# 10. CONVERT CATEGORIES TO NUMBERS
# ============================================================

category_mapping = {
    "Very Low": 0,
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Very High": 4
}


category_columns = [
    "WRI_Category",
    "Exposure_Category",
    "Vulnerability_Category",
    "Susceptibility_Category"
]


for col in category_columns:

    df[col] = (
        df[col]
        .astype(str)
        .map(category_mapping)
    )


# ============================================================
# 11. CHECK CLASS DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)

for col in category_columns:

    print("\n", col)

    print(
        df[col].value_counts(
            sort=False
        )
    )


# ============================================================
# 12. SELECT FEATURES
# ============================================================

# IMPORTANT:
#
# We do NOT use WRI, Exposure, Vulnerability,
# or Susceptibility themselves as input features
# when predicting their corresponding categories.
#
# We use the underlying indicators instead.


target_columns = [
    "WRI_Category",
    "Exposure_Category",
    "Vulnerability_Category",
    "Susceptibility_Category"
]


# Main aggregate score columns
# are removed from features to avoid direct leakage.

columns_to_remove = [
    "Country",
    "ISO3",
    "WRI",
    "Exposure",
    "Vulnerability",
    "Susceptibility",
    "WRI_Category",
    "Exposure_Category",
    "Vulnerability_Category",
    "Susceptibility_Category"
]


feature_columns = [
    col
    for col in df.columns
    if col not in columns_to_remove
]


# Keep numeric features only

feature_columns = [
    col
    for col in feature_columns
    if pd.api.types.is_numeric_dtype(
        df[col]
    )
]


print("\nNumber of features:", len(feature_columns))

print("\nFirst 30 features:")

print(feature_columns[:30])


# ============================================================
# 13. CREATE X
# ============================================================

X = df[
    feature_columns
].copy()


# ============================================================
# 14. HANDLE INFINITE VALUES
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# 15. FILL MISSING FEATURE VALUES
# ============================================================

for col in X.columns:

    if X[col].isna().any():

        X[col] = X[col].fillna(
            X[col].median()
        )


# ============================================================
# 16. CREATE TARGET DATA
# ============================================================

y = df[
    target_columns
].copy()


# Remove any rows where category creation failed

valid_rows = ~y.isna().any(axis=1)

X = X.loc[
    valid_rows
].reset_index(drop=True)

y = y.loc[
    valid_rows
].reset_index(drop=True)


print("\nFinal X shape:")
print(X.shape)

print("\nFinal y shape:")
print(y.shape)


# ============================================================
# 17. TRAIN / TEST SPLIT
# ============================================================

# The paper uses:
#
# 80% training
# 20% independent test
#
# The paper also uses stratified 5-fold
# cross-validation during model development.


# Use WRI category for stratification.

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y["WRI_Category"]
)


print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print("\nTraining samples:", len(X_train))
print("Testing samples :", len(X_test))


# ============================================================
# 18. SAVE CLEANED DATASET
# ============================================================

cleaned_file = os.path.join(
    PROCESSED_DIR,
    "worldriskindex_cleaned.csv"
)

df.to_csv(
    cleaned_file,
    index=False
)


# ============================================================
# 19. SAVE TRAIN / TEST DATA
# ============================================================

X_train.to_csv(
    os.path.join(
        PROCESSED_DIR,
        "X_train.csv"
    ),
    index=False
)

X_test.to_csv(
    os.path.join(
        PROCESSED_DIR,
        "X_test.csv"
    ),
    index=False
)

y_train.to_csv(
    os.path.join(
        PROCESSED_DIR,
        "y_train.csv"
    ),
    index=False
)

y_test.to_csv(
    os.path.join(
        PROCESSED_DIR,
        "y_test.csv"
    ),
    index=False
)


# ============================================================
# 20. SAVE FEATURE INFORMATION
# ============================================================

feature_information = {

    "features": feature_columns,

    "targets": target_columns,

    "category_mapping": category_mapping,

    "category_names": category_names
}


joblib.dump(
    feature_information,
    os.path.join(
        MODEL_DIR,
        "feature_information.pkl"
    )
)


# ============================================================
# 21. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 60)
print("PHASE 2 COMPLETED SUCCESSFULLY")
print("=" * 60)

print("\nSaved files:")

print(
    "data/processed/worldriskindex_cleaned.csv"
)

print(
    "data/processed/X_train.csv"
)

print(
    "data/processed/X_test.csv"
)

print(
    "data/processed/y_train.csv"
)

print(
    "data/processed/y_test.csv"
)

print(
    "models/feature_information.pkl"
)

print("\nTargets:")

for target in target_columns:
    print(" -", target)

print("\nNumber of features:", len(feature_columns))

print("\nTraining samples:", len(X_train))

print("Testing samples :", len(X_test))

print("\nREADY FOR PHASE 3")

PHASE 2 - DATA PREPROCESSING

Loading dataset:
c:\Users\mvars\Downloads\open_lab\data\raw\worldriskindex-trend.csv

Dataset loaded successfully.
Rows   : 5018
Columns: 248

First 20 columns:
['WRI.Country', 'ISO3.Code', 'Year', 'W', 'E', 'V', 'S', 'C', 'A', 'S_01', 'S_02', 'S_03', 'S_04', 'S_05', 'C_01', 'C_02', 'C_03', 'A_01', 'A_02', 'A_03']

Required columns found.

Main columns after renaming:
['Country', 'ISO3', 'Year', 'WRI', 'Exposure', 'Vulnerability', 'Susceptibility', 'Coping', 'Adaptive']

Missing values:
Year              0
WRI               0
Exposure          0
Vulnerability     0
Susceptibility    0
Coping            0
Adaptive          0
dtype: int64

Rows removed: 0
Rows remaining: 5018

CLASS DISTRIBUTION

 WRI_Category
WRI_Category
2    1002
3    1003
0    1008
4    1004
1    1001
Name: count, dtype: int64

 Exposure_Category
Exposure_Category
1    1019
3    1001
0    1014
2     980
4    1004
Name: count, dtype: int64

 Vulnerability_Category
Vulnerability_Category
4